###Transform Customer Data
1. Remove records with NULL customer_id
2. Remove exact duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the columns to the correct data type
5. Write transformed data to the Silver schema

###1. Remove records with NULL customer_id

In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL;

###2. Remove exact duplicate records

In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct
AS
SELECT DISTINCT * FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
SELECT customer_id, 
      MAX(created_timestamp) AS max_created 
FROM v_customers_distinct
group by customer_id;

###3. Remove duplicate records based on created_timestamp

In [0]:
%sql
WITH cte_max AS
(
    SELECT customer_id, 
           MAX(created_timestamp) AS max_created 
FROM v_customers_distinct
group by customer_id
)
SELECT v_customers_distinct.*
FROM v_customers_distinct
inner join cte_max
on v_customers_distinct.customer_id = cte_max.customer_id
and v_customers_distinct.created_timestamp  = cte_max.max_created 

###4. CAST the columns to the correct data type

In [0]:
%sql
WITH cte_max AS
(
    SELECT customer_id, 
           MAX(created_timestamp) AS max_created 
FROM v_customers_distinct
group by customer_id
)
SELECT CAST(t.created_timestamp AS timestamp) AS ts,
       t.customer_id,
       t.customer_name,
       CAST(t.date_of_birth AS date) AS date_of_birth,
       t.email,
       CAST(t.member_since AS date) AS member_since,
       t.telephone
FROM v_customers_distinct t
inner join cte_max m
on t.customer_id = m.customer_id
and t.created_timestamp  = m.max_created 

###5. Write transformed data to the Silver schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.customers
WITH cte_max AS
(
    SELECT customer_id, 
           MAX(created_timestamp) AS max_created 
FROM v_customers_distinct
group by customer_id
)
SELECT CAST(t.created_timestamp AS timestamp) AS ts,
       t.customer_id,
       t.customer_name,
       CAST(t.date_of_birth AS date) AS date_of_birth,
       t.email,
       CAST(t.member_since AS date) AS member_since,
       t.telephone
FROM v_customers_distinct t
inner join cte_max m
on t.customer_id = m.customer_id
and t.created_timestamp  = m.max_created

In [0]:
%sql
SELECT * FROM gizmobox.silver.customers;

In [0]:
%sql
DESCRIBE EXTENDED gizmobox.silver.customers;